# Gunicorn - Python WSGI HTTP Server

## Example 1: Basic Gunicorn Configuration and Settings

In [ ]:
import gunicorn
from gunicorn.config import Config
from gunicorn.app.base import BaseApplication
import os

print("Gunicorn - Python WSGI HTTP Server")
print(f"Gunicorn version: {gunicorn.__version__}")

# Create a basic configuration
config = Config()

# Test default configuration values
print(f"\nDefault Configuration:")
print(f"Bind address: {config.bind}")
print(f"Worker processes: {config.workers}")
print(f"Worker class: {config.worker_class}")
print(f"Worker timeout: {config.timeout}")
print(f"Keep alive: {config.keepalive}")
print(f"Max requests: {config.max_requests}")

# Test configuration validation
print(f"\nConfiguration validation:")
try:
    # Test setting various config options
    test_config = {
        'bind': '127.0.0.1:8000',
        'workers': 2,
        'worker_class': 'sync',
        'timeout': 30,
        'keepalive': 2
    }
    
    for key, value in test_config.items():
        if hasattr(config, key):
            print(f"✓ {key}: {value} (valid setting)")
        else:
            print(f"✗ {key}: Unknown setting")
            
except Exception as e:
    print(f"Configuration error: {e}")
    raise

print(f"\nGunicorn configuration test completed!")

## Example 2: WSGI Application Integration

In [ ]:
from gunicorn.app.base import BaseApplication
import tempfile
import os

# Create a simple WSGI application for testing
def simple_wsgi_app(environ, start_response):
    """A simple WSGI application for testing"""
    status = '200 OK'
    headers = [('Content-Type', 'text/plain')]
    start_response(status, headers)
    
    # Return information about the request
    response_data = []
    response_data.append(f"Method: {environ['REQUEST_METHOD']}")
    response_data.append(f"Path: {environ['PATH_INFO']}")
    response_data.append(f"Query: {environ.get('QUERY_STRING', '')}")
    response_data.append(f"Server: {environ.get('SERVER_NAME', 'unknown')}")
    response_data.append(f"Port: {environ.get('SERVER_PORT', 'unknown')}")
    
    return [('\n'.join(response_data)).encode('utf-8')]

# Test WSGI application wrapper
class TestGunicornApp(BaseApplication):
    def __init__(self, app, options=None):
        self.options = options or {}
        self.application = app
        super().__init__()
    
    def load_config(self):
        for key, value in self.options.items():
            self.cfg.set(key.lower(), value)
    
    def load(self):
        return self.application

print("WSGI Application Integration Test")

# Create test configuration
test_options = {
    'bind': '127.0.0.1:0',  # Use port 0 to get any available port
    'workers': 1,
    'worker_class': 'sync',
    'timeout': 10,
    'preload_app': True
}

# Create the Gunicorn application wrapper
app = TestGunicornApp(simple_wsgi_app, test_options)

print(f"✓ WSGI application created")
print(f"✓ Configuration loaded: {len(test_options)} options")
print(f"✓ Application callable: {callable(app.application)}")

# Test WSGI environ simulation
test_environ = {
    'REQUEST_METHOD': 'GET',
    'PATH_INFO': '/test',
    'QUERY_STRING': 'param=value',
    'SERVER_NAME': 'localhost',
    'SERVER_PORT': '8000'
}

def test_start_response(status, headers):
    print(f"Response status: {status}")
    print(f"Response headers: {len(headers)} headers")

# Test the WSGI app
result = simple_wsgi_app(test_environ, test_start_response)
print(f"\nWSGI response: {result[0].decode()}")

## Example 3: Worker Process Configuration

In [ ]:
from gunicorn.workers import sync, gthread
from gunicorn.config import Config
import multiprocessing

print("Gunicorn Worker Process Configuration")

# Test worker class information
worker_classes = [
    ('sync', 'Synchronous worker'),
    ('gthread', 'Threaded worker'),
    ('gevent', 'Gevent async worker'),
    ('eventlet', 'Eventlet async worker')
]

print("Available worker classes:")
for worker_class, description in worker_classes:
    print(f"  {worker_class}: {description}")

# Test optimal worker count calculation
cpu_count = multiprocessing.cpu_count()
recommended_workers = (cpu_count * 2) + 1

print(f"\nWorker Configuration Recommendations:")
print(f"CPU cores: {cpu_count}")
print(f"Recommended workers: {recommended_workers}")
print(f"Formula: (CPU cores * 2) + 1")

# Test worker configuration options
worker_configs = [
    {'workers': 1, 'worker_class': 'sync', 'threads': 1},
    {'workers': 2, 'worker_class': 'gthread', 'threads': 2},
    {'workers': 4, 'worker_class': 'sync', 'threads': 1},
]

print(f"\nWorker Configuration Examples:")
for i, config in enumerate(worker_configs, 1):
    print(f"Config {i}: {config['workers']} workers, {config['worker_class']} class, {config['threads']} threads")
    
    # Calculate theoretical request capacity
    if config['worker_class'] == 'gthread':
        capacity = config['workers'] * config['threads']
    else:
        capacity = config['workers']
    
    print(f"  Concurrent request capacity: {capacity}")

# Test worker resource limits
print(f"\nWorker Resource Limits:")
resource_limits = {
    'max_requests': 1000,
    'max_requests_jitter': 50,
    'timeout': 30,
    'keepalive': 2,
    'worker_connections': 1000
}

for limit_name, limit_value in resource_limits.items():
    print(f"  {limit_name}: {limit_value}")

print(f"\nWorker configuration analysis completed!")

## Example 4: Logging and Monitoring Integration

In [ ]:
from gunicorn.glogging import Logger
from gunicorn.config import Config
import logging
import tempfile
import os

print("Gunicorn Logging and Monitoring")

# Test Gunicorn logger configuration
config = Config()

# Test log level settings
log_levels = ['debug', 'info', 'warning', 'error', 'critical']
print("Available log levels:")
for level in log_levels:
    numeric_level = getattr(logging, level.upper(), None)
    if numeric_level:
        print(f"  {level}: {numeric_level}")

# Test access log format
print(f"\nAccess Log Format Options:")
access_log_formats = {
    'combined': '%(h)s %(l)s %(u)s %(t)s "%(r)s" %(s)s %(b)s "%(f)s" "%(a)s"',
    'common': '%(h)s %(l)s %(u)s %(t)s "%(r)s" %(s)s %(b)s',
    'simple': '%(h)s "%(r)s" %(s)s %(b)s'
}

for format_name, format_string in access_log_formats.items():
    print(f"  {format_name}: {format_string[:50]}...")

# Test log configuration
with tempfile.TemporaryDirectory() as temp_dir:
    access_log = os.path.join(temp_dir, "access.log")
    error_log = os.path.join(temp_dir, "error.log")
    
    log_config = {
        'accesslog': access_log,
        'errorlog': error_log,
        'loglevel': 'info',
        'access_log_format': '%(h)s "%(r)s" %(s)s %(b)s'
    }
    
    print(f"\nLog Configuration Test:")
    for key, value in log_config.items():
        print(f"  {key}: {value}")
    
    # Test log file creation: gunicorn's Logger opens the files the config names
    for key, value in log_config.items():
        config.set(key, value)
    logger = Logger(config)
    logger.info("gunicorn logging example: info")
    logger.error("gunicorn logging example: error")
    
    print(f"\n✓ Log files created successfully")
    print(f"  Access log: {os.path.exists(access_log)}")
    print(f"  Error log: {os.path.exists(error_log)}")
    assert os.path.exists(access_log) and os.path.exists(error_log), os.listdir(temp_dir)
    with open(error_log) as f:
        error_log_text = f.read()
    # measured, gunicorn 26.2.0: one "[<time>] [<pid>] [<LEVEL>] <message>" line per record
    assert "[INFO] gunicorn logging example: info" in error_log_text, error_log_text
    assert "[ERROR] gunicorn logging example: error" in error_log_text, error_log_text
    for handler in logger.error_log.handlers + logger.access_log.handlers:
        handler.close()

# Test monitoring metrics
print(f"\nMonitoring Metrics:")
metrics = [
    'worker_processes',
    'active_requests', 
    'total_requests',
    'failed_requests',
    'response_time',
    'memory_usage',
    'cpu_usage'
]

for metric in metrics:
    print(f"  - {metric}")

print(f"\nLogging and monitoring configuration completed!")

## Example 5: Error Handling and Signal Management

In [ ]:
import signal
import os
import time
from gunicorn.config import Config

print("Gunicorn Error Handling and Signal Management")

# Test signal handling
signals_info = [
    ('SIGTERM', signal.SIGTERM, 'Graceful shutdown'),
    ('SIGINT', signal.SIGINT, 'Interrupt (Ctrl+C)'),
    ('SIGQUIT', signal.SIGQUIT, 'Quick shutdown'),
    ('SIGHUP', signal.SIGHUP, 'Reload configuration'),
    ('SIGUSR1', signal.SIGUSR1, 'Reopen log files'),
    ('SIGUSR2', signal.SIGUSR2, 'Upgrade worker processes')
]

print("Signal handling capabilities:")
for sig_name, sig_num, description in signals_info:
    print(f"  {sig_name} ({sig_num}): {description}")

# Test error handling scenarios
print(f"\nError Handling Scenarios:")
error_scenarios = [
    "Worker timeout - process hangs",
    "Memory exhaustion - worker crash",
    "Import error - application fails to load",
    "Port binding error - address already in use",
    "Permission error - can't bind to privileged port",
    "Configuration error - invalid settings"
]

for i, scenario in enumerate(error_scenarios, 1):
    print(f"  {i}. {scenario}")

# Test graceful shutdown simulation
print(f"\nGraceful Shutdown Process:")
shutdown_steps = [
    "1. Receive SIGTERM signal",
    "2. Stop accepting new requests",
    "3. Wait for active requests to complete",
    "4. Timeout after grace period",
    "5. Force terminate remaining workers",
    "6. Clean up resources and exit"
]

for step in shutdown_steps:
    print(f"  {step}")

# Test configuration validation
print(f"\nConfiguration Validation:")
try:
    config = Config()
    
    # Test invalid configurations
    invalid_configs = [
        {'workers': 0, 'error': 'Workers must be > 0'},
        {'bind': 'invalid:port', 'error': 'Invalid bind address'},
        {'timeout': -1, 'error': 'Timeout must be positive'},
        {'worker_class': 'nonexistent', 'error': 'Unknown worker class'}
    ]
    
    for test_config in invalid_configs:
        error_msg = test_config.pop('error')
        print(f"  ✗ {test_config}: {error_msg}")
        
except Exception as e:
    print(f"Configuration validation error: {e}")
    raise

# Test process monitoring
print(f"\nProcess Monitoring:")
process_info = {
    'PID': os.getpid(),
    'Parent PID': os.getppid(),
    'Process group': os.getpgrp(),
    'Session ID': os.getsid(0)
}

for info_name, info_value in process_info.items():
    print(f"  {info_name}: {info_value}")

print(f"\nError handling and signal management test completed!")